# Module 2 — Deploy to AgentCore Runtime

In Module 1 your Chief of Staff agent ran on your laptop. Great for building — but no one else can reach
it. In this module you take the **exact same agent** and deploy it to **Amazon Bedrock AgentCore
Runtime**: a managed, serverless runtime that runs your agent in an isolated microVM and exposes it over
HTTP.

You'll deploy it **bare** — just the agent, in production. **No memory, no observability dashboards yet.**
That's deliberate: by the end you'll hit a real limitation (the agent forgets you between calls) that
**Module 3** fixes.

### What you'll do

| Step | What happens |
|------|--------------|
| **1. Recap the reuse** | See how the deployed agent reuses Module 1's `build_agent_options()` — one source of truth |
| **2. The entrypoint** | Understand the thin `@app.entrypoint` wrapper |
| **3. Configure** | Point AgentCore at your AWS account; review `agentcore.json` |
| **4. Test locally** | `agentcore dev` runs the container locally |
| **5. Deploy** | `agentcore deploy` builds the image, pushes to ECR, creates the runtime |
| **6. Invoke** | Call your live agent over HTTP and see the response |
| **7. Hit the wall** | Notice it's stateless — the hook into Module 3 |
| **8. Clean up** | Tear down so nothing keeps billing |

## Why AgentCore Runtime?

Getting an agent to production normally means building session isolation, scaling, credential management,
and an HTTP layer yourself. AgentCore Runtime gives you all of that with a single deploy:

| Feature | What you get |
|---|---|
| **Isolated microVM per session** | Each session runs in its own sandbox |
| **Serverless & auto-scaling** | No servers to manage; scales with demand |
| **Managed identity/credentials** | The runtime gets a scoped IAM role automatically |
| **HTTP protocol** | Invoke over HTTP; streaming supported |

You bring the agent; AgentCore runs it.

## Step 1 — One agent, two front doors (the reuse)

We are **not** rewriting the agent. Module 1 and Module 2 share **one source of truth** for the agent's
identity — `build_agent_options()` in `agent.py`:

- **Module 1 (local):** `send_query()` calls `build_agent_options()` and runs the agent in-process.
- **Module 2 (deploy):** `agent_agentcore.py` calls the *same* `build_agent_options()` inside an AgentCore
  entrypoint.

Run the cell below to see that the deployment entrypoint contains **no agent logic of its own** — it just
wraps and reuses.

In [ ]:
import sys
sys.path.insert(0, "chief_of_staff_agent")

import inspect
from agent import build_agent_options
import agent_agentcore

# The entrypoint imports the shared identity builder...
src = inspect.getsource(agent_agentcore)
assert "from agent import build_agent_options" in src
assert "build_agent_options()" in src
# ...and does NOT redefine the system prompt or tool list.
assert "system_prompt" not in src.replace("build_agent_options", "")
print("✅ agent_agentcore.py reuses build_agent_options() — no duplicated agent logic")

opts = build_agent_options()
print("   tools:", opts.allowed_tools)
print("   setting_sources:", opts.setting_sources)

## Step 2 — The AgentCore entrypoint

AgentCore runs your agent through a small **entrypoint**: a handler that receives a request payload and
streams a response. Here is the whole thing (`chief_of_staff_agent/agent_agentcore.py`):

```python
from bedrock_agentcore import BedrockAgentCoreApp
from claude_agent_sdk import ClaudeSDKClient
from agent import build_agent_options          # ← reuse Module 1's identity

app = BedrockAgentCoreApp()

@app.entrypoint
async def invoke(payload: dict):
    prompt = (payload or {}).get("prompt")
    options = build_agent_options()             # ← same config as local
    async with ClaudeSDKClient(options=options) as agent:
        await agent.query(prompt)
        async for msg in agent.receive_response():
            for block in getattr(msg, "content", []) or []:
                if getattr(block, "text", None):
                    yield block.text             # ← stream text back

if __name__ == "__main__":
    app.run()                                    # serves /invocations + /ping on :8080
```

`BedrockAgentCoreApp` implements the runtime's HTTP contract (`/invocations`, `/ping`) for you. The Module
1 agent logic moves inside this handler **unchanged**, because it's the same `build_agent_options()`.

## Why a Container build (not a zip)?

AgentCore supports two build types: **CodeZip** (Python zipped to S3) and **Container** (a Docker image).
We use **Container**, and here's the concrete reason:

> The Claude Agent SDK ships a ~218MB **native CLI binary**. Packaged as a zip, that binary arrives in the
> runtime **without its execute permission**, and the agent dies at startup with
> `Permission denied: .../claude_agent_sdk/_bundled/claude`.

A Container build `pip install`s the SDK **inside a Linux/ARM64 image**, so the binary has the right
architecture and permissions. The `Dockerfile` lives in `chief_of_staff_agent/`. (AgentCore Runtime
requires `linux/arm64` images.)

## Step 3 — Configure the deployment

The deployment is declared in `agentcore/agentcore.json` (already set up for you). Two things you must
personalize, because they depend on **your** AWS account:

1. **Deployment target** — your account ID + region:

   ```bash
   cp agentcore/aws-targets.example.json agentcore/aws-targets.json
   # edit aws-targets.json: "account" = your 12-digit ID, "region" = your region
   ```

2. **Environment / model** — already set in `agentcore.json` `envVars` (Bedrock model IDs). Adjust if you
   use different models.

Then validate the config:

In [ ]:
import subprocess
print(subprocess.run(["agentcore", "validate"], capture_output=True, text=True).stdout)

Let's look at what we're about to deploy — the runtime entry in `agentcore.json`:

In [ ]:
import json
cfg = json.load(open("agentcore/agentcore.json"))
print(json.dumps(cfg["runtimes"][0], indent=2))
# Note: build=Container, protocol=HTTP, enableOtel=true (traces → CloudWatch, explored in Module 4).
# The execution IAM role (Bedrock invoke + CloudWatch Logs + X-Ray) is created automatically by the CDK.

## Step 4 — Test locally first (`agentcore dev`)

Before deploying to AWS, run the agent locally in a container that mimics the runtime. This is the fast
dev loop. In a **terminal** (it's a long-running server):

```bash
agentcore dev                     # builds + runs the container locally on :8080
```

Then, in another terminal, invoke it locally:

```bash
agentcore dev "What is our current monthly burn rate?"
```

You should see the Chief of Staff answer using the company data — exactly like Module 1, but now running
through the AgentCore HTTP contract.

## Step 5 — Deploy to AgentCore Runtime

This builds the container image, pushes it to ECR, and provisions the runtime via CDK. It takes a few
minutes and creates real AWS resources. Run in a **terminal**:

```bash
agentcore deploy -y
```

When it finishes you'll get a runtime ARN. Check status anytime:

```bash
agentcore status
```

In [ ]:
# You can also check status from the notebook once deployed:
import subprocess
out = subprocess.run(["agentcore", "status"], capture_output=True, text=True)
print(out.stdout or out.stderr)

## Step 6 — Invoke your deployed agent

The agent is now live and reachable over HTTP. Invoke it:

In [ ]:
import subprocess, json

result = subprocess.run(
    ["agentcore", "invoke", "What is our current runway and cash position?"],
    capture_output=True, text=True,
)
print(result.stdout or result.stderr)
# The response comes from your agent running in AgentCore Runtime — not your laptop.

That response was produced by your agent **running in AWS**, using the same `CLAUDE.md` company context
and the `financial-analysis` skill you built in Module 1 — now served from managed infrastructure.

> **Observability note:** traces for this invocation are already flowing to CloudWatch (we turned on
> `enableOtel`). We'll *explore* them in **Module 4**.

## Step 7 — The limitation you'll hit (hook into Module 3)

Try invoking twice, where the second call depends on the first:

```bash
agentcore invoke "My name is Sarah and I'm the CEO."
agentcore invoke "What's my name?"
```

The agent **won't remember**. Each invocation is independent — a fresh, isolated session. That's what
"stateless" means, and it's the right default for an auto-scaling runtime. But real products need to
remember their users.

::: That's exactly what **Module 3 — Add AgentCore Memory** fixes. :::

## Step 8 — Clean up

To avoid ongoing charges, tear down the runtime when you're done. Run in a **terminal**:

```bash
agentcore remove agent --name cos     # remove from config
agentcore deploy -y                   # apply removal → destroys the runtime/stack
```

Confirm nothing lingers:

In [ ]:
import subprocess
out = subprocess.run(["agentcore", "status"], capture_output=True, text=True)
print(out.stdout or out.stderr)

## Key takeaways

- AgentCore Runtime gives you isolated, auto-scaling, managed hosting with a single `agentcore deploy`.
- You deployed your **existing** agent by wrapping it in a thin entrypoint that **reuses
  `build_agent_options()`** — no duplicated agent logic.
- The Claude Agent SDK's bundled binary means **Container builds** (Linux/ARM64), not zip.
- The CDK auto-creates the **execution IAM role**; `enableOtel` already ships traces to CloudWatch.
- A bare deployment is **stateless** — which is exactly what **Module 3 (Memory)** addresses next.

## Next steps

Continue to **Module 3 — Add AgentCore Memory** to make your deployed agent remember users across calls.